In [370]:
# Cell 1

import os
import sys
import json
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nNotebook 03 environment initialized successfully.")

Python executable:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\venv\Scripts\python.exe

Python version:
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]

Notebook 03 environment initialized successfully.


In [371]:
# Cell 2

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"

RAG_INDEX_DIR = OUTPUT_DIR / "rag_index"
EXTRACTED_ENTITIES_DIR = OUTPUT_DIR / "extracted_entities"
COMPARISON_DIR = OUTPUT_DIR / "comparison"
REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [
    OUTPUT_DIR,
    RAG_INDEX_DIR,
    EXTRACTED_ENTITIES_DIR,
    COMPARISON_DIR,
    REPORTS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nOutput directories created:")
print("RAG index:", RAG_INDEX_DIR)
print("Extracted entities:", EXTRACTED_ENTITIES_DIR)
print("Comparison:", COMPARISON_DIR)
print("Reports:", REPORTS_DIR)

Project root:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant

Processed directory:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\data\processed

Output directories created:
RAG index: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\rag_index
Extracted entities: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\extracted_entities
Comparison: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\comparison
Reports: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\reports


In [372]:
# Cell 3

# Search the project for architecture_knowledge.json
matches = list(
    PROJECT_ROOT.rglob("architecture_knowledge.json")
)

if not matches:

    raise FileNotFoundError(
        "architecture_knowledge.json was not found anywhere "
        "inside the project:\n"
        f"{PROJECT_ROOT}\n\n"
        "Please make sure Notebook 02 was executed and "
        "the architecture knowledge JSON was saved."
    )

print("Architecture knowledge file found:")

for path in matches:
    print("-", path)

# Use the first matching file
architecture_file = matches[0]

print("\nSelected file:")
print(architecture_file)

print(
    "\nFile size:",
    architecture_file.stat().st_size,
    "bytes"
)

Architecture knowledge file found:
- c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\extracted_entities\architecture_knowledge.json

Selected file:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\extracted_entities\architecture_knowledge.json

File size: 11308 bytes


In [373]:
# Cell 4

with open(
    architecture_file,
    "r",
    encoding="utf-8"
) as f:

    architecture_knowledge = json.load(f)

print("Architecture knowledge loaded successfully.")

print("\nTop-level keys:")

for key in architecture_knowledge.keys():
    print("-", key)

Architecture knowledge loaded successfully.

Top-level keys:
- document
- components
- ports
- interfaces
- data_flows
- functional_flows
- dependencies
- architecture_relationships


In [374]:
# Cell 5

expected_entity_types = [
    "components",
    "ports",
    "interfaces",
    "data_flows",
    "functional_flows",
    "dependencies",
    "architecture_relationships"
]

print("=" * 70)
print("AUTOSAR ARCHITECTURE KNOWLEDGE VALIDATION")
print("=" * 70)

for entity_type in expected_entity_types:

    value = architecture_knowledge.get(
        entity_type,
        []
    )

    if isinstance(value, list):

        print(
            f"{entity_type:30s}: {len(value)}"
        )

    else:

        print(
            f"{entity_type:30s}: "
            f"Unexpected type "
            f"({type(value).__name__})"
        )

print("=" * 70)

print(
    "\nDocument:",
    architecture_knowledge.get(
        "document",
        "Unknown"
    )
)

AUTOSAR ARCHITECTURE KNOWLEDGE VALIDATION
components                    : 3
ports                         : 7
interfaces                    : 3
data_flows                    : 5
functional_flows              : 6
dependencies                  : 3
architecture_relationships    : 3

Document: sample_autosar_hld.pdf


In [375]:
# Cell 6

rag_documents = []

document_name = architecture_knowledge.get(
    "document",
    "unknown_document"
)

print("RAG document collection initialized.")
print("Source document:", document_name)
print("Current document count:", len(rag_documents))

RAG document collection initialized.
Source document: sample_autosar_hld.pdf
Current document count: 0


In [376]:
# Cell 7

for component in architecture_knowledge.get(
    "components",
    []
):

    component_name = component.get(
        "name",
        "Unknown Component"
    )

    role = component.get(
        "role",
        ""
    )

    inputs = component.get(
        "inputs",
        []
    )

    outputs = component.get(
        "outputs",
        []
    )

    text = (
        f"Component: {component_name}\n"
        f"Role: {role}\n"
        f"Inputs: {', '.join(map(str, inputs))}\n"
        f"Outputs: {', '.join(map(str, outputs))}"
    )

    rag_documents.append({
        "id": f"component_{component_name}",
        "type": "component",
        "text": text,
        "source": document_name,
        "page": component.get("page"),
        "section": component.get("section"),
        "metadata": component
    })

print(
    "Component documents added:",
    sum(
        1
        for doc in rag_documents
        if doc["type"] == "component"
    )
)

print(
    "Total RAG documents:",
    len(rag_documents)
)

Component documents added: 3
Total RAG documents: 3


In [377]:
# Cell 8

for interface in architecture_knowledge.get(
    "interfaces",
    []
):

    interface_name = interface.get(
        "interface",
        interface.get(
            "name",
            "Unknown Interface"
        )
    )

    provider = interface.get(
        "provider",
        ""
    )

    consumer = interface.get(
        "consumer",
        ""
    )

    data_elements = interface.get(
        "data_elements",
        []
    )

    text = (
        f"Interface: {interface_name}\n"
        f"Provider: {provider}\n"
        f"Consumer: {consumer}\n"
        f"Data Elements: "
        f"{', '.join(map(str, data_elements))}"
    )

    rag_documents.append({
        "id": f"interface_{interface_name}",
        "type": "interface",
        "text": text,
        "source": document_name,
        "page": interface.get("page"),
        "section": interface.get("section"),
        "metadata": interface
    })

print(
    "Interface documents added:",
    sum(
        1
        for doc in rag_documents
        if doc["type"] == "interface"
    )
)

print(
    "Total RAG documents:",
    len(rag_documents)
)

Interface documents added: 3
Total RAG documents: 6


In [378]:
# Cell 9

for flow in architecture_knowledge.get(
    "data_flows",
    []
):

    signal = flow.get(
        "signal",
        "Unknown Signal"
    )

    relationship = flow.get(
        "relationship",
        ""
    )

    component = flow.get(
        "component",
        ""
    )

    text = (
        f"Signal: {signal}\n"
        f"Relationship: {relationship}\n"
        f"Component: {component}"
    )

    rag_documents.append({
        "id": (
            f"data_flow_{signal}_"
            f"{relationship}_"
            f"{component}"
        ),
        "type": "data_flow",
        "text": text,
        "source": document_name,
        "page": flow.get("page"),
        "section": flow.get("section"),
        "metadata": flow
    })

print(
    "Data-flow documents added:",
    sum(
        1
        for doc in rag_documents
        if doc["type"] == "data_flow"
    )
)

print(
    "Total RAG documents:",
    len(rag_documents)
)

Data-flow documents added: 5
Total RAG documents: 11


In [379]:
# Cell 10

for dependency in architecture_knowledge.get(
    "dependencies",
    []
):

    dependent = dependency.get(
        "dependent_component",
        ""
    )

    dependency_component = dependency.get(
        "dependency_component",
        ""
    )

    relationship = dependency.get(
        "relationship",
        ""
    )

    text = (
        f"Dependent component: {dependent}\n"
        f"Dependency component: "
        f"{dependency_component}\n"
        f"Relationship: {relationship}"
    )

    rag_documents.append({
        "id": (
            f"dependency_"
            f"{dependent}_"
            f"{dependency_component}"
        ),
        "type": "dependency",
        "text": text,
        "source": document_name,
        "page": dependency.get("page"),
        "section": dependency.get("section"),
        "metadata": dependency
    })

print(
    "Dependency documents added:",
    sum(
        1
        for doc in rag_documents
        if doc["type"] == "dependency"
    )
)

print(
    "Total RAG documents:",
    len(rag_documents)
)

Dependency documents added: 3
Total RAG documents: 14


In [380]:
# Cell 11

for flow in architecture_knowledge.get(
    "functional_flows",
    []
):

    step = flow.get(
        "step",
        ""
    )

    description = flow.get(
        "description",
        ""
    )

    text = (
        f"Step {step}: "
        f"{description}"
    )

    rag_documents.append({
        "id": f"functional_flow_{step}",
        "type": "functional_flow",
        "text": text,
        "source": document_name,
        "page": flow.get("page"),
        "section": flow.get("section"),
        "metadata": flow
    })

functional_count = sum(
    1
    for doc in rag_documents
    if doc["type"] == "functional_flow"
)

print(
    "Functional-flow documents added:",
    functional_count
)

print(
    "Total RAG documents:",
    len(rag_documents)
)

Functional-flow documents added: 6
Total RAG documents: 20


In [381]:
# Cell 12

for relationship in architecture_knowledge.get(
    "architecture_relationships",
    []
):

    text = (
        f"Architecture relationship: "
        f"{relationship}"
    )

    rag_documents.append({
        "id": (
            "architecture_relationship_"
            + str(
                len(
                    [
                        doc
                        for doc in rag_documents
                        if doc["type"]
                        == "architecture_relationship"
                    ]
                )
            )
        ),
        "type": "architecture_relationship",
        "text": text,
        "source": document_name,
        "page": relationship.get("page")
        if isinstance(relationship, dict)
        else None,
        "section": relationship.get("section")
        if isinstance(relationship, dict)
        else None,
        "metadata": relationship
    })

relationship_count = sum(
    1
    for doc in rag_documents
    if doc["type"]
    == "architecture_relationship"
)

print(
    "Architecture relationship documents added:",
    relationship_count
)

print(
    "Total RAG documents:",
    len(rag_documents)
)

Architecture relationship documents added: 3
Total RAG documents: 23


In [382]:
# Cell 13

required_fields = [
    "id",
    "type",
    "text",
    "source",
    "metadata"
]

invalid_documents = []

for index, document in enumerate(
    rag_documents
):

    missing_fields = [
        field
        for field in required_fields
        if field not in document
    ]

    if missing_fields:

        invalid_documents.append({
            "index": index,
            "missing": missing_fields
        })

print("=" * 70)
print("RAG DOCUMENT VALIDATION")
print("=" * 70)

print(
    "Total documents:",
    len(rag_documents)
)

print(
    "Invalid documents:",
    len(invalid_documents)
)

if invalid_documents:

    print("\nProblems found:")

    for item in invalid_documents:
        print(item)

    raise ValueError(
        "RAG document validation failed."
    )

print("\nAll RAG documents are valid.")

RAG DOCUMENT VALIDATION
Total documents: 23
Invalid documents: 0

All RAG documents are valid.


In [383]:
# Cell 14

print("=" * 70)
print("RAG DOCUMENT SAMPLE")
print("=" * 70)

for document in rag_documents[:5]:

    print(
        "\nID:",
        document["id"]
    )

    print(
        "Type:",
        document["type"]
    )

    print(
        "Text:"
    )

    print(
        document["text"]
    )

    print(
        "Source:",
        document["source"]
    )

    print("-" * 70)

RAG DOCUMENT SAMPLE

ID: component_BatteryMonitor
Type: component
Text:
Component: BatteryMonitor
Role: Battery measurement and status
Inputs: BatteryVoltage, BatteryTemperature
Outputs: BatteryStatus
Source: sample_autosar_hld.pdf
----------------------------------------------------------------------

ID: component_ThermalController
Type: component
Text:
Component: ThermalController
Role: Thermal management
Inputs: BatteryTemperature
Outputs: CoolingRequest
Source: sample_autosar_hld.pdf
----------------------------------------------------------------------

ID: component_Dashboard
Type: component
Text:
Component: Dashboard
Role: Driver display
Inputs: BatteryStatus
Outputs: DisplayUpdate
Source: sample_autosar_hld.pdf
----------------------------------------------------------------------

ID: interface_BatteryMeasurementInterface
Type: interface
Text:
Interface: BatteryMeasurementInterface
Provider: BatteryMonitor
Consumer: ThermalController
Data Elements: BatteryVoltage, BatteryTemp

In [384]:
# Cell 15

rag_documents_file = (
    RAG_INDEX_DIR /
    "rag_documents.json"
)

with open(
    rag_documents_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rag_documents,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print(
    "RAG documents saved successfully."
)

print(
    "\nFile:",
    rag_documents_file
)

print(
    "Documents saved:",
    len(rag_documents)
)

print(
    "File size:",
    rag_documents_file.stat().st_size,
    "bytes"
)

RAG documents saved successfully.

File: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\rag_index\rag_documents.json
Documents saved: 23
File size: 13827 bytes


In [385]:
# Cell 16

from fastembed import TextEmbedding

embedding_model_name = "BAAI/bge-small-en-v1.5"

embedding_model = TextEmbedding(
    model_name=embedding_model_name
)

print("Embedding model loaded successfully.")
print("Model:", embedding_model_name)

Embedding model loaded successfully.
Model: BAAI/bge-small-en-v1.5


In [386]:
# Cell 17

document_texts = [
    document["text"]
    for document in rag_documents
]

embeddings = list(
    embedding_model.embed(
        document_texts
    )
)

print(
    "Documents embedded:",
    len(embeddings)
)

if embeddings:

    print(
        "Embedding dimension:",
        len(embeddings[0])
    )

    print(
        "First embedding type:",
        type(embeddings[0])
    )

Documents embedded: 23
Embedding dimension: 384
First embedding type: <class 'numpy.ndarray'>


In [387]:
# Cell 18

import numpy as np
import faiss

embedding_matrix = np.asarray(
    embeddings,
    dtype="float32"
)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(
    embedding_matrix
)

embedding_dimension = (
    embedding_matrix.shape[1]
)

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(
    embedding_matrix
)

print("FAISS index created successfully.")

print(
    "Embedding dimension:",
    embedding_dimension
)

print(
    "Indexed documents:",
    faiss_index.ntotal
)

FAISS index created successfully.
Embedding dimension: 384
Indexed documents: 23


In [388]:
# Cell 19

faiss_index_file = (
    RAG_INDEX_DIR /
    "autosar_hld.faiss"
)

faiss.write_index(
    faiss_index,
    str(faiss_index_file)
)

print(
    "FAISS index saved successfully."
)

print(
    "\nFile:",
    faiss_index_file
)

print(
    "File size:",
    faiss_index_file.stat().st_size,
    "bytes"
)

FAISS index saved successfully.

File: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\rag_index\autosar_hld.faiss
File size: 35373 bytes


In [389]:
# Cell 20

def semantic_search(
    query,
    top_k=5
):

    # Convert query into embedding
    query_embedding = list(
        embedding_model.embed(
            [query]
        )
    )[0]

    query_vector = np.asarray(
        [query_embedding],
        dtype="float32"
    )

    # Normalize query vector
    faiss.normalize_L2(
        query_vector
    )

    # Search FAISS
    scores, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for score, index in zip(
        scores[0],
        indices[0]
    ):

        if index < 0:
            continue

        document = rag_documents[index].copy()

        document["score"] = float(score)

        results.append(
            document
        )

    return results


print(
    "Semantic search function created successfully."
)

Semantic search function created successfully.


In [390]:
# Cell 21

test_query = "Which component consumes BatteryTemperature?"

search_results = semantic_search(
    test_query,
    top_k=5
)

print("=" * 70)
print("SEMANTIC SEARCH TEST")
print("=" * 70)

print("Query:", test_query)
print("Results:", len(search_results))

for i, result in enumerate(
    search_results,
    start=1
):

    print("\n" + "-" * 70)

    print(
        f"Result {i}"
    )

    print(
        "Type:",
        result.get("type")
    )

    print(
        "Score:",
        round(
            result.get("score", 0),
            4
        )
    )

    print(
        "Text:",
        result.get("text")
    )

    print(
        "Source:",
        result.get("source")
    )

    print(
        "Page:",
        result.get("page")
    )

    print(
        "Section:",
        result.get("section")
    )

SEMANTIC SEARCH TEST
Query: Which component consumes BatteryTemperature?
Results: 5

----------------------------------------------------------------------
Result 1
Type: data_flow
Score: 0.8516
Text: Signal: BatteryTemperature
Relationship: consumed
Component: ThermalController
Source: sample_autosar_hld.pdf
Page: 4
Section: None

----------------------------------------------------------------------
Result 2
Type: dependency
Score: 0.8243
Text: Dependent component: ThermalController
Dependency component: BatteryMonitor
Relationship: 
Source: sample_autosar_hld.pdf
Page: 4
Section: None

----------------------------------------------------------------------
Result 3
Type: data_flow
Score: 0.8174
Text: Signal: BatteryStatus
Relationship: consumed
Component: Dashboard
Source: sample_autosar_hld.pdf
Page: 4
Section: None

----------------------------------------------------------------------
Result 4
Type: interface
Score: 0.815
Text: Interface: BatteryMeasurementInterface
Provider: Batt

In [391]:
# Cell 22

def find_data_flows(signal):
    return [
        flow
        for flow in architecture_knowledge.get(
            "data_flows", []
        )
        if str(flow.get("signal", "")).lower()
        == signal.lower()
    ]


def find_dependencies(component):
    return [
        dependency
        for dependency in architecture_knowledge.get(
            "dependencies", []
        )
        if str(
            dependency.get(
                "dependency_component", ""
            )
        ).lower()
        == component.lower()
    ]


def find_interface_between(component_a, component_b):

    matches = []

    for interface in architecture_knowledge.get(
        "interfaces", []
    ):

        provider = str(
            interface.get("provider", "")
        ).lower()

        consumer = str(
            interface.get("consumer", "")
        ).lower()

        a = component_a.lower()
        b = component_b.lower()

        if (
            (provider == a and consumer == b)
            or
            (provider == b and consumer == a)
        ):

            matches.append(interface)

    return matches


def find_functional_flows():
    return architecture_knowledge.get(
        "functional_flows", []
    )


print("Structured lookup functions created successfully.")

Structured lookup functions created successfully.


In [392]:
# Cell 23

signal = "BatteryTemperature"

flow_results = find_data_flows(
    signal
)

print("=" * 70)
print("STRUCTURED DATA-FLOW TEST")
print("=" * 70)

print(
    "Signal:",
    signal
)

print(
    "Matching records:",
    len(flow_results)
)

for result in flow_results:

    print("\n", result)

STRUCTURED DATA-FLOW TEST
Signal: BatteryTemperature
Matching records: 2

 {'signal': 'BatteryTemperature', 'relationship': 'produced', 'component': 'BatteryMonitor', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '7', 'section_title': 'Signals and Data Flow'}

 {'signal': 'BatteryTemperature', 'relationship': 'consumed', 'component': 'ThermalController', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '7', 'section_title': 'Signals and Data Flow'}


In [393]:
# Cell 24

component = "BatteryMonitor"

dependency_results = find_dependencies(
    component
)

print("=" * 70)
print("STRUCTURED DEPENDENCY TEST")
print("=" * 70)

print(
    "Component:",
    component
)

print(
    "Matching dependencies:",
    len(dependency_results)
)

for result in dependency_results:

    print("\n", result)

STRUCTURED DEPENDENCY TEST
Component: BatteryMonitor
Matching dependencies: 2

 {'dependent_component': 'ThermalController', 'dependency_component': 'BatteryMonitor', 'data_element': 'BatteryTemperature', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '9', 'section_title': 'Dependency Mapping'}

 {'dependent_component': 'Dashboard', 'dependency_component': 'BatteryMonitor', 'data_element': 'BatteryStatus', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '9', 'section_title': 'Dependency Mapping'}


In [394]:
# Cell 25

component_a = "BatteryMonitor"
component_b = "Dashboard"

interface_results = find_interface_between(
    component_a,
    component_b
)

print("=" * 70)
print("STRUCTURED INTERFACE TEST")
print("=" * 70)

print(
    f"Components: {component_a} <-> {component_b}"
)

print(
    "Matching interfaces:",
    len(interface_results)
)

for result in interface_results:

    print("\n", result)

STRUCTURED INTERFACE TEST
Components: BatteryMonitor <-> Dashboard
Matching interfaces: 1

 {'name': 'BatteryStatusInterface', 'type': 'Interface', 'data_elements': ['BatteryStatus'], 'provider': 'BatteryMonitor', 'consumer': 'Dashboard', 'document': 'sample_autosar_hld.pdf', 'page': 3, 'section_number': '6', 'section_title': 'Interface Definitions'}


In [395]:
# Cell 26

def combined_retrieval(query, top_k=5):

    query_lower = query.lower()

    results = []

    # --------------------------------------------------
    # 1. Semantic retrieval
    # --------------------------------------------------

    results.extend(
        semantic_search(
            query,
            top_k=top_k
        )
    )

    # --------------------------------------------------
    # 2. Exact signal lookup
    # --------------------------------------------------

    known_signals = [
        "BatteryVoltage",
        "BatteryTemperature",
        "BatteryStatus",
        "CoolingRequest",
        "DisplayUpdate"
    ]

    for signal in known_signals:

        if signal.lower() in query_lower:

            for flow in find_data_flows(signal):

                results.append({
                    "id": (
                        "structured_data_flow_"
                        + signal
                    ),
                    "type": "data_flow",
                    "text": (
                        f"Signal {signal} is "
                        f"{flow.get('relationship')} "
                        f"by {flow.get('component')}."
                    ),
                    "source": flow.get("document"),
                    "page": flow.get("page"),
                    "section": flow.get("section_title"),
                    "metadata": {
                        "signal": flow.get("signal"),
                        "relationship": flow.get("relationship"),
                        "component": flow.get("component")
                    }
                })

    # --------------------------------------------------
    # 3. Provider lookup
    # --------------------------------------------------

    if (
        "provides" in query_lower
        or "produces" in query_lower
    ):

        for interface in architecture_knowledge.get(
            "interfaces",
            []
        ):

            elements = [
                str(element).lower()
                for element in interface.get(
                    "data_elements",
                    []
                )
            ]

            provider = interface.get("provider")

            if not provider:
                continue

            for element in elements:

                if element in query_lower:

                    results.append({
                        "id": (
                            "provider_"
                            + str(interface.get("name"))
                        ),
                        "type": "interface",
                        "text": (
                            f"{provider} provides "
                            f"{element} through "
                            f"{interface.get('name')}."
                        ),
                        "source": interface.get("document"),
                        "page": interface.get("page"),
                        "section": interface.get(
                            "section_title"
                        ),
                        "metadata": {
                            "name": interface.get("name"),
                            "provider": provider,
                            "consumer": interface.get(
                                "consumer"
                            ),
                            "data_elements":
                                interface.get(
                                    "data_elements"
                                )
                        }
                    })

    # --------------------------------------------------
    # 4. Dependency lookup
    # --------------------------------------------------

    if "depend" in query_lower:

        for dependency in architecture_knowledge.get(
            "dependencies",
            []
        ):

            dependent = dependency.get(
                "dependent_component"
            )

            dependency_component = dependency.get(
                "dependency_component"
            )

            results.append({
                "id": (
                    "dependency_"
                    + str(dependent)
                    + "_"
                    + str(dependency_component)
                ),
                "type": "dependency",
                "text": (
                    f"{dependent} depends on "
                    f"{dependency_component}."
                ),
                "source": dependency.get("document"),
                "page": dependency.get("page"),
                "section": dependency.get(
                    "section_title"
                ),
                "metadata": {
                    "dependent_component": dependent,
                    "dependency_component":
                        dependency_component
                }
            })

    # --------------------------------------------------
    # 5. Interface lookup
    # --------------------------------------------------

    if "interface" in query_lower:

        components = [
            "BatteryMonitor",
            "ThermalController",
            "Dashboard",
            "CoolingSystem"
        ]

        mentioned = [
            component
            for component in components
            if component.lower() in query_lower
        ]

        if len(mentioned) >= 2:

            interfaces = find_interface_between(
                mentioned[0],
                mentioned[1]
            )

            for interface in interfaces:

                results.append({
                    "id": (
                        "interface_"
                        + str(interface.get("name"))
                    ),
                    "type": "interface",
                    "text": (
                        f"{interface.get('name')} "
                        f"connects "
                        f"{interface.get('provider')} "
                        f"to "
                        f"{interface.get('consumer')}."
                    ),
                    "source": interface.get("document"),
                    "page": interface.get("page"),
                    "section": interface.get(
                        "section_title"
                    ),
                    "metadata": {
                        "name": interface.get("name"),
                        "provider": interface.get(
                            "provider"
                        ),
                        "consumer": interface.get(
                            "consumer"
                        ),
                        "data_elements":
                            interface.get(
                                "data_elements"
                            )
                    }
                })

    # --------------------------------------------------
    # 6. Complete functional-flow retrieval
    # --------------------------------------------------

    if "flow" in query_lower:

        functional_flows = find_functional_flows()

        for index, flow in enumerate(
            functional_flows
        ):

            if isinstance(flow, dict):

                step = flow.get(
                    "step",
                    index + 1
                )

                description = flow.get(
                    "description",
                    flow.get(
                        "flow",
                        str(flow)
                    )
                )

                page = flow.get("page")

                section = flow.get(
                    "section_title"
                )

                metadata = flow

            else:

                step = index + 1
                description = str(flow)
                page = None
                section = None
                metadata = {
                    "step": step,
                    "description": description
                }

            results.append({
                "id":
                    f"functional_flow_{step}",

                "type":
                    "functional_flow",

                "text":
                    f"Step {step}: {description}",

                "source":
                    document_name,

                "page":
                    page,

                "section":
                    section,

                "metadata":
                    metadata
            })

    return results


print(
    "Combined retrieval function updated successfully."
)

Combined retrieval function updated successfully.


In [396]:
# Cell 27

def deduplicate_evidence(results):
    """
    Remove duplicate architecture evidence.

    Handles both:
    1. Semantic RAG records
    2. Structured architecture records
    """

    unique_results = []
    seen = set()

    for result in results:

        metadata = result.get("metadata", {})

        result_type = result.get("type")

        # --------------------------------------------------
        # Dependency
        # --------------------------------------------------
        if result_type == "dependency":

            key = (
                "dependency",
                str(
                    metadata.get(
                        "dependent_component",
                        ""
                    )
                ).lower(),
                str(
                    metadata.get(
                        "dependency_component",
                        ""
                    )
                ).lower()
            )

        # --------------------------------------------------
        # Data flow
        # --------------------------------------------------
        elif result_type == "data_flow":

            key = (
                "data_flow",
                str(
                    metadata.get(
                        "signal",
                        ""
                    )
                ).lower(),
                str(
                    metadata.get(
                        "relationship",
                        ""
                    )
                ).lower(),
                str(
                    metadata.get(
                        "component",
                        ""
                    )
                ).lower()
            )

        # --------------------------------------------------
        # Interface
        # --------------------------------------------------
        elif result_type == "interface":

            key = (
                "interface",
                str(
                    metadata.get(
                        "name",
                        ""
                    )
                ).lower()
            )

        # --------------------------------------------------
        # Component
        # --------------------------------------------------
        elif result_type == "component":

            key = (
                "component",
                str(
                    metadata.get(
                        "name",
                        ""
                    )
                ).lower()
            )

        # --------------------------------------------------
        # Functional flow
        # --------------------------------------------------
        elif result_type == "functional_flow":

            key = (
                "functional_flow",
                str(
                    result.get(
                        "text",
                        ""
                    )
                ).strip().lower()
            )

        # --------------------------------------------------
        # Architecture relationship
        # --------------------------------------------------
        elif result_type == "architecture_relationship":

            key = (
                "architecture_relationship",
                str(
                    result.get(
                        "text",
                        ""
                    )
                ).strip().lower()
            )

        # --------------------------------------------------
        # Fallback
        # --------------------------------------------------
        else:

            key = (
                result_type,
                str(
                    result.get(
                        "text",
                        ""
                    )
                ).strip().lower()
            )

        if key not in seen:

            seen.add(key)
            unique_results.append(result)

    return unique_results


print("Improved evidence deduplication function created successfully.")

Improved evidence deduplication function created successfully.


In [397]:
# Cell 28

def filter_relevant_evidence(query, results):
    """
    Filter retrieved evidence according to the user's question.
    Keeps architecture evidence from multiple valid sources.
    """

    query_lower = query.lower()

    # --------------------------------------------------
    # Consumes
    # --------------------------------------------------

    if "consumes" in query_lower:

        filtered = []

        for result in results:

            if result.get("type") != "data_flow":
                continue

            metadata = result.get(
                "metadata",
                {}
            )

            signal = str(
                metadata.get(
                    "signal",
                    ""
                )
            ).lower()

            relationship = str(
                metadata.get(
                    "relationship",
                    ""
                )
            ).lower()

            if (
                signal
                and signal in query_lower
                and relationship == "consumed"
            ):

                filtered.append(result)

        return filtered

    # --------------------------------------------------
    # Provides / Produces
    # --------------------------------------------------

    if (
        "provides" in query_lower
        or "produces" in query_lower
    ):

        filtered = []

        # Find known architecture signals
        known_signals = [
            "batteryvoltage",
            "batterytemperature",
            "batterystatus",
            "coolingrequest",
            "displayupdate"
        ]

        requested_signals = [
            signal
            for signal in known_signals
            if signal in query_lower
        ]

        for result in results:

            result_type = result.get(
                "type"
            )

            metadata = result.get(
                "metadata",
                {}
            )

            text = str(
                result.get(
                    "text",
                    ""
                )
            ).lower()

            # ------------------------------------------
            # Component outputs
            # ------------------------------------------

            if result_type == "component":

                outputs = [
                    str(output).lower()
                    for output in metadata.get(
                        "outputs",
                        []
                    )
                ]

                if any(
                    signal in outputs
                    for signal in requested_signals
                ):

                    filtered.append(result)

            # ------------------------------------------
            # Interface provider
            # ------------------------------------------

            elif result_type == "interface":

                provider = metadata.get(
                    "provider"
                )

                elements = [
                    str(element).lower()
                    for element in metadata.get(
                        "data_elements",
                        []
                    )
                ]

                if (
                    provider
                    and any(
                        signal in elements
                        for signal in requested_signals
                    )
                ):

                    filtered.append(result)

            # ------------------------------------------
            # Functional-flow evidence
            # ------------------------------------------

            elif result_type == "functional_flow":

                if (
                    "provides" in text
                    and any(
                        signal in text
                        for signal in requested_signals
                    )
                ):

                    filtered.append(result)

            # ------------------------------------------
            # Produced data-flow evidence
            # ------------------------------------------

            elif result_type == "data_flow":

                relationship = str(
                    metadata.get(
                        "relationship",
                        ""
                    )
                ).lower()

                signal = str(
                    metadata.get(
                        "signal",
                        ""
                    )
                ).lower()

                if (
                    signal in requested_signals
                    and relationship in [
                        "produced",
                        "generated"
                    ]
                ):

                    filtered.append(result)

        return filtered

    # --------------------------------------------------
    # Dependencies
    # --------------------------------------------------

    if "depend" in query_lower:

        return [
            result
            for result in results
            if result.get(
                "type"
            ) == "dependency"
        ]

    # --------------------------------------------------
    # Interfaces
    # --------------------------------------------------

    if "interface" in query_lower:

        return [
            result
            for result in results
            if result.get(
                "type"
            ) == "interface"
        ]

    # --------------------------------------------------
    # Functional flows
    # --------------------------------------------------

    if "flow" in query_lower:

        return [
            result
            for result in results
            if result.get(
                "type"
            ) == "functional_flow"
        ]

    # --------------------------------------------------
    # Generic query
    # --------------------------------------------------

    return results


print(
    "Corrected evidence filtering function created successfully."
)

Corrected evidence filtering function created successfully.


In [398]:
# Cell 29

def prepare_evidence(query, top_k=5):
    """
    Prepare grounded evidence for an AUTOSAR HLD question.

    Functional-flow questions retrieve the complete
    ordered flow instead of relying only on top-k
    semantic retrieval.
    """

    query_lower = query.lower()

    # --------------------------------------------------
    # Functional-flow questions
    # --------------------------------------------------

    if "flow" in query_lower:

        functional_flows = find_functional_flows()

        flow_evidence = []

        for index, flow in enumerate(
            functional_flows
        ):

            if isinstance(flow, dict):

                step = flow.get(
                    "step",
                    index + 1
                )

                description = flow.get(
                    "description",
                    flow.get(
                        "flow",
                        str(flow)
                    )
                )

                page = flow.get(
                    "page"
                )

                section = flow.get(
                    "section_title"
                )

                metadata = flow.copy()

            else:

                step = index + 1
                description = str(flow)
                page = None
                section = None

                metadata = {
                    "step": step,
                    "description": description
                }

            flow_evidence.append({
                "id":
                    f"functional_flow_{step}",

                "type":
                    "functional_flow",

                "text":
                    f"Step {step}: {description}",

                "source":
                    document_name,

                "page":
                    page,

                "section":
                    section,

                "metadata":
                    metadata
            })

        # Preserve documented sequence
        flow_evidence.sort(
            key=lambda item:
                item.get(
                    "metadata",
                    {}
                ).get(
                    "step",
                    999
                )
        )

        return flow_evidence

    # --------------------------------------------------
    # Normal RAG retrieval
    # --------------------------------------------------

    results = combined_retrieval(
        query,
        top_k=top_k
    )

    results = deduplicate_evidence(
        results
    )

    results = filter_relevant_evidence(
        query,
        results
    )

    results = deduplicate_evidence(
        results
    )

    return results


print(
    "Improved evidence preparation function created successfully."
)

Improved evidence preparation function created successfully.


In [399]:
# Cell 30

test_queries = [

    "Which component consumes BatteryTemperature?",

    "Which components depend on BatteryMonitor?",

    "What interface connects BatteryMonitor and Dashboard?"
]

for query in test_queries:

    evidence = prepare_evidence(
        query
    )

    print("=" * 70)
    print("QUESTION:")
    print(query)

    print(
        "\nEvidence records:",
        len(evidence)
    )

    for index, result in enumerate(
        evidence,
        start=1
    ):

        print(
            f"\nEvidence {index}"
        )

        print(
            "Type:",
            result.get("type")
        )

        print(
            "Text:",
            result.get("text")
        )

        print(
            "Source:",
            result.get("source")
        )

        print(
            "Page:",
            result.get("page")
        )

        print(
            "Section:",
            result.get("section")
        )

QUESTION:
Which component consumes BatteryTemperature?

Evidence records: 1

Evidence 1
Type: data_flow
Text: Signal: BatteryTemperature
Relationship: consumed
Component: ThermalController
Source: sample_autosar_hld.pdf
Page: 4
Section: None
QUESTION:
Which components depend on BatteryMonitor?

Evidence records: 3

Evidence 1
Type: dependency
Text: Dependent component: ThermalController
Dependency component: BatteryMonitor
Relationship: 
Source: sample_autosar_hld.pdf
Page: 4
Section: None

Evidence 2
Type: dependency
Text: Dependent component: Dashboard
Dependency component: BatteryMonitor
Relationship: 
Source: sample_autosar_hld.pdf
Page: 4
Section: None

Evidence 3
Type: dependency
Text: CoolingSystem depends on ThermalController.
Source: sample_autosar_hld.pdf
Page: 4
Section: Dependency Mapping
QUESTION:
What interface connects BatteryMonitor and Dashboard?

Evidence records: 1

Evidence 1
Type: interface
Text: Interface: BatteryStatusInterface
Provider: BatteryMonitor
Consumer: 

In [400]:
# Cell 31

def format_citation(result):

    source = result.get(
        "source",
        "Unknown document"
    )

    page = result.get("page")

    section = result.get("section")

    citation = str(source)

    if page is not None:
        citation += f", Page {page}"

    if section:
        citation += f", Section {section}"

    return citation


print("Citation formatter created successfully.")

Citation formatter created successfully.


In [401]:
# Cell 32

def format_evidence_for_llm(
    evidence
):

    if not evidence:
        return (
            "No relevant evidence was retrieved."
        )

    evidence_blocks = []

    for index, result in enumerate(
        evidence,
        start=1
    ):

        block = (
            f"Evidence {index}\n"
            f"Type: {result.get('type', '')}\n"
            f"Content:\n"
            f"{result.get('text', '')}\n"
            f"Source: "
            f"{format_citation(result)}"
        )

        evidence_blocks.append(
            block
        )

    return "\n\n".join(
        evidence_blocks
    )


print(
    "LLM evidence formatter created successfully."
)

LLM evidence formatter created successfully.


In [402]:
# Cell 33

from openai import OpenAI

openai_api_key = os.getenv(
    "OPENAI_API_KEY"
)

if not openai_api_key:

    print(
        "WARNING: OPENAI_API_KEY is not set."
    )

    print(
        "The RAG retrieval pipeline can still "
        "be tested, but OpenAI generation will "
        "not run until the key is configured."
    )

    openai_client = None

else:

    openai_client = OpenAI(
        api_key=openai_api_key
    )

    print(
        "OpenAI client initialized successfully."
    )

OpenAI client initialized successfully.


In [403]:
# Cell 34

def build_grounded_prompt(
    query,
    evidence
):

    formatted_evidence = (
        format_evidence_for_llm(
            evidence
        )
    )

    system_prompt = """
You are an AUTOSAR High-Level Design
Document Analysis Assistant.

Your task is to answer questions using
ONLY the retrieved evidence provided to you.

Rules:

1. Do not invent architecture information.
2. Do not use information that is not present
   in the retrieved evidence.
3. Preserve component, interface, port,
   signal, and flow names exactly.
4. If the evidence is insufficient, clearly say:
   "The provided evidence is insufficient
   to answer the question."
5. Every factual answer must include the
   provided source citation.
6. Do not modify or approve architecture decisions.
7. Your response is an engineering aid and
   must remain subject to human review.
"""

    user_prompt = f"""
Question:
{query}

Retrieved HLD Evidence:
{formatted_evidence}

Answer the question using only this evidence.
"""

    return system_prompt, user_prompt


print(
    "Grounded OpenAI prompt builder created successfully."
)

Grounded OpenAI prompt builder created successfully.


In [404]:
# Cell 35

test_query = (
    "Which component consumes BatteryTemperature?"
)

test_evidence = prepare_evidence(
    test_query
)

system_prompt, user_prompt = (
    build_grounded_prompt(
        test_query,
        test_evidence
    )
)

print("=" * 70)
print("GROUNDED PROMPT TEST")
print("=" * 70)

print("\nSYSTEM PROMPT:")
print(system_prompt)

print("\nUSER PROMPT:")
print(user_prompt)

GROUNDED PROMPT TEST

SYSTEM PROMPT:

You are an AUTOSAR High-Level Design
Document Analysis Assistant.

Your task is to answer questions using
ONLY the retrieved evidence provided to you.

Rules:

1. Do not invent architecture information.
2. Do not use information that is not present
   in the retrieved evidence.
3. Preserve component, interface, port,
   signal, and flow names exactly.
4. If the evidence is insufficient, clearly say:
   "The provided evidence is insufficient
   to answer the question."
5. Every factual answer must include the
   provided source citation.
6. Do not modify or approve architecture decisions.
7. Your response is an engineering aid and
   must remain subject to human review.


USER PROMPT:

Question:
Which component consumes BatteryTemperature?

Retrieved HLD Evidence:
Evidence 1
Type: data_flow
Content:
Signal: BatteryTemperature
Relationship: consumed
Component: ThermalController
Source: sample_autosar_hld.pdf, Page 4

Answer the question using only th

In [405]:
# Cell 36

def generate_openai_answer(
    query,
    evidence,
    model="gpt-6-astra"
):

    if not evidence:

        return {
            "answer": (
                "The provided evidence is insufficient "
                "to answer the question."
            ),
            "model": None,
            "used_openai": False
        }

    if openai_client is None:

        return {
            "answer": (
                "OpenAI API key is not configured. "
                "Retrieved evidence is available, "
                "but LLM generation cannot be performed."
            ),
            "model": None,
            "used_openai": False
        }

    system_prompt, user_prompt = (
        build_grounded_prompt(
            query,
            evidence
        )
    )

    try:

        response = openai_client.responses.create(
            model=model,
            instructions=system_prompt,
            input=user_prompt
        )

        return {
            "answer": response.output_text,
            "model": model,
            "used_openai": True
        }

    except Exception as e:

        return {
            "answer": (
                "OpenAI generation failed: "
                f"{type(e).__name__}: {str(e)}"
            ),
            "model": model,
            "used_openai": False
        }


print(
    "OpenAI generation function created successfully."
)

print(
    "Generation model: gpt-6-astra"
)

OpenAI generation function created successfully.
Generation model: gpt-6-astra


In [406]:
# Cell 37

def answer_question(
    query,
    top_k=5,
    model="gpt-6-astra"
):

    evidence = prepare_evidence(
        query,
        top_k=top_k
    )

    result = generate_openai_answer(
        query,
        evidence,
        model=model
    )

    return {
        "query": query,
        "answer": result["answer"],
        "evidence": evidence,
        "model": result["model"],
        "used_openai": result["used_openai"]
    }


print(
    "End-to-end RAG answer function created successfully."
)

print(
    "Default model: gpt-6-astra"
)

End-to-end RAG answer function created successfully.
Default model: gpt-6-astra


In [407]:
# Cell 38

query = (
    "Which component consumes BatteryTemperature?"
)

evidence = prepare_evidence(
    query
)

print("=" * 70)
print("RETRIEVED EVIDENCE")
print("=" * 70)

print(
    format_evidence_for_llm(
        evidence
    )
)

RETRIEVED EVIDENCE
Evidence 1
Type: data_flow
Content:
Signal: BatteryTemperature
Relationship: consumed
Component: ThermalController
Source: sample_autosar_hld.pdf, Page 4


In [408]:
# Cell 39

if openai_client is None:

    print(
        "OpenAI client is not configured."
    )

    print(
        "Skipping API test."
    )

else:

    try:

        response = openai_client.responses.create(
            model="gpt-6-astra",
            input="Reply with exactly: OPENAI_CONNECTION_OK"
        )

        print(
            "OpenAI connection successful."
        )

        print(
            "Response:"
        )

        print(
            response.output_text
        )

    except Exception as e:

        print(
            "OpenAI API test failed."
        )

        print(
            "Error type:",
            type(e).__name__
        )

        print(
            "Error:",
            str(e)
        )

OpenAI API test failed.
Error type: RateLimitError
Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


In [409]:
# Cell 40

query = "Which component consumes BatteryTemperature?"

result = answer_question(
    query,
    model="gpt-6-astra"
)

print("=" * 70)
print("AUTOSAR HLD RAG ASSISTANT")
print("=" * 70)

print("\nQuestion:")
print(result["query"])

print("\nRetrieved Evidence:")
print(
    format_evidence_for_llm(
        result["evidence"]
    )
)

print("\nAnswer:")

if result["used_openai"]:

    print(result["answer"])

else:

    print(
        "OpenAI generation is unavailable "
        "because the API account has no remaining credits."
    )

    print("\nEvidence-grounded answer:")

    for evidence in result["evidence"]:

        metadata = evidence.get(
            "metadata",
            {}
        )

        if (
            evidence.get("type") == "data_flow"
            and metadata.get("relationship")
            == "consumed"
        ):

            print(
                f"{metadata.get('component')} "
                f"consumes "
                f"{metadata.get('signal')}."
            )

            print(
                "Source:",
                format_citation(evidence)
            )

print(
    "\nEvidence used:",
    len(result["evidence"])
)

print(
    "OpenAI used:",
    result["used_openai"]
)

print(
    "Model:",
    result["model"]
)

AUTOSAR HLD RAG ASSISTANT

Question:
Which component consumes BatteryTemperature?

Retrieved Evidence:
Evidence 1
Type: data_flow
Content:
Signal: BatteryTemperature
Relationship: consumed
Component: ThermalController
Source: sample_autosar_hld.pdf, Page 4

Answer:
OpenAI generation is unavailable because the API account has no remaining credits.

Evidence-grounded answer:
ThermalController consumes BatteryTemperature.
Source: sample_autosar_hld.pdf, Page 4

Evidence used: 1
OpenAI used: False
Model: gpt-6-astra


In [410]:
# Cell 41

def generate_local_answer(query, evidence):

    if not evidence:
        return (
            "Insufficient evidence was retrieved "
            "from the HLD."
        )

    query_lower = query.lower()

    answers = []

    # --------------------------------------------------
    # Consumes
    # --------------------------------------------------

    if "consumes" in query_lower:

        for item in evidence:

            if item.get("type") != "data_flow":
                continue

            metadata = item.get(
                "metadata",
                {}
            )

            if metadata.get(
                "relationship"
            ) == "consumed":

                answers.append(
                    f"{metadata.get('component')} "
                    f"consumes "
                    f"{metadata.get('signal')}. "
                    f"[{format_citation(item)}]"
                )

    # --------------------------------------------------
    # Provides / Produces
    # --------------------------------------------------

    elif (
        "provides" in query_lower
        or "produces" in query_lower
    ):

        for item in evidence:

            metadata = item.get(
                "metadata",
                {}
            )

            # Interface provider
            if item.get("type") == "interface":

                provider = metadata.get(
                    "provider"
                )

                elements = metadata.get(
                    "data_elements",
                    []
                )

                if provider:

                    for element in elements:

                        if str(
                            element
                        ).lower() in query_lower:

                            answers.append(
                                f"{provider} provides "
                                f"{element} through "
                                f"{metadata.get('name')}. "
                                f"[{format_citation(item)}]"
                            )

            # Data-flow producer
            elif item.get(
                "type"
            ) == "data_flow":

                if metadata.get(
                    "relationship"
                ) in [
                    "produced",
                    "generated"
                ]:

                    answers.append(
                        f"{metadata.get('component')} "
                        f"provides "
                        f"{metadata.get('signal')}. "
                        f"[{format_citation(item)}]"
                    )

    # --------------------------------------------------
    # Dependencies
    # --------------------------------------------------

    elif "depend" in query_lower:

        for item in evidence:

            if item.get(
                "type"
            ) != "dependency":

                continue

            metadata = item.get(
                "metadata",
                {}
            )

            answers.append(
                f"{metadata.get('dependent_component')} "
                f"depends on "
                f"{metadata.get('dependency_component')}. "
                f"[{format_citation(item)}]"
            )

    # --------------------------------------------------
    # Interface
    # --------------------------------------------------

    elif "interface" in query_lower:

        for item in evidence:

            if item.get(
                "type"
            ) != "interface":

                continue

            metadata = item.get(
                "metadata",
                {}
            )

            answers.append(
                f"{metadata.get('name')} connects "
                f"{metadata.get('provider')} to "
                f"{metadata.get('consumer')}. "
                f"[{format_citation(item)}]"
            )

    # --------------------------------------------------
    # Functional flow
    # --------------------------------------------------

    elif "flow" in query_lower:

        flow_steps = []

        for item in evidence:

            if item.get(
                "type"
            ) != "functional_flow":

                continue

            flow_steps.append(
                item
            )

        # Sort by documented step number
        flow_steps.sort(
            key=lambda item:
                item.get(
                    "metadata",
                    {}
                ).get(
                    "step",
                    999
                )
        )

        for item in flow_steps:

            text = item.get(
                "text",
                ""
            )

            if text:

                answers.append(
                    f"{text} "
                    f"[{format_citation(item)}]"
                )

    # --------------------------------------------------
    # Generic fallback
    # --------------------------------------------------

    else:

        for item in evidence[:3]:

            answers.append(
                f"{item.get('text')} "
                f"[{format_citation(item)}]"
            )

    if not answers:

        return (
            "The retrieved HLD evidence does not contain "
            "enough information to answer this question."
        )

    return "\n".join(answers)


print(
    "Local evidence-grounded answer generator updated successfully."
)

Local evidence-grounded answer generator updated successfully.


In [411]:
# Cell 42

def unified_answer(query, top_k=5, model="gpt-6-astra"):
    """
    Answer an AUTOSAR HLD question.

    Uses OpenAI when available.
    Otherwise uses the local evidence-grounded generator.
    """

    evidence = prepare_evidence(
        query,
        top_k=top_k
    )

    if openai_client is not None:

        result = generate_openai_answer(
            query=query,
            evidence=evidence,
            model=model
        )

        if result.get("used_openai"):

            return {
                "query": query,
                "answer": result["answer"],
                "evidence": evidence,
                "used_openai": True,
                "model": model
            }

    # Local fallback

    local_answer = generate_local_answer(
        query,
        evidence
    )

    return {
        "query": query,
        "answer": local_answer,
        "evidence": evidence,
        "used_openai": False,
        "model": "local-evidence-fallback"
    }


print("Unified answer function created successfully.")

Unified answer function created successfully.


In [412]:
# Cell 43

test_questions = [
    "Which component consumes BatteryTemperature?",
    "Which components depend on BatteryMonitor?",
    "What interface connects BatteryMonitor and Dashboard?",
    "Which component provides BatteryStatus?",
    "What is the functional flow from battery measurement to cooling request?"
]

print("=" * 70)
print("MULTI-QUESTION RAG TEST")
print("=" * 70)

for number, question in enumerate(
    test_questions,
    start=1
):

    result = unified_answer(question)

    print(f"\nQuestion {number}:")
    print(question)

    print("\nAnswer:")
    print(result["answer"])

    print(
        "\nEvidence count:",
        len(result["evidence"])
    )

    print(
        "Generation mode:",
        result["model"]
    )

    print("-" * 70)

MULTI-QUESTION RAG TEST

Question 1:
Which component consumes BatteryTemperature?

Answer:
ThermalController consumes BatteryTemperature. [sample_autosar_hld.pdf, Page 4]

Evidence count: 1
Generation mode: local-evidence-fallback
----------------------------------------------------------------------

Question 2:
Which components depend on BatteryMonitor?

Answer:
ThermalController depends on BatteryMonitor. [sample_autosar_hld.pdf, Page 4]
Dashboard depends on BatteryMonitor. [sample_autosar_hld.pdf, Page 4]
CoolingSystem depends on ThermalController. [sample_autosar_hld.pdf, Page 4, Section Dependency Mapping]

Evidence count: 3
Generation mode: local-evidence-fallback
----------------------------------------------------------------------

Question 3:
What interface connects BatteryMonitor and Dashboard?

Answer:
BatteryStatusInterface connects BatteryMonitor to Dashboard. [sample_autosar_hld.pdf, Page 3]

Evidence count: 1
Generation mode: local-evidence-fallback
-------------------

In [413]:
# Cell 44

def validate_citations(result):
    """
    Validate whether retrieved evidence contains
    usable source information.
    """

    evidence = result.get("evidence", [])

    validation = {
        "answer_present": bool(result.get("answer")),
        "evidence_present": len(evidence) > 0,
        "valid_citations": 0,
        "missing_citations": 0,
        "citation_coverage": 0.0
    }

    for item in evidence:

        citation = format_citation(item)

        if citation != "Unknown document":
            validation["valid_citations"] += 1
        else:
            validation["missing_citations"] += 1

    if evidence:
        validation["citation_coverage"] = (
            validation["valid_citations"] / len(evidence)
        )

    return validation


print("Citation validation function created successfully.")

Citation validation function created successfully.


In [414]:
# Cell 45

evaluation_questions = [
    {
        "question": "Which component consumes BatteryTemperature?",
        "expected": "ThermalController"
    },
    {
        "question": "Which components depend on BatteryMonitor?",
        "expected": "ThermalController"
    },
    {
        "question": "What interface connects BatteryMonitor and Dashboard?",
        "expected": "BatteryStatusInterface"
    },
    {
        "question": "Which component provides BatteryStatus?",
        "expected": "BatteryMonitor"
    },
    {
        "question": "What is the functional flow from battery measurement to cooling request?",
        "expected": "CoolingRequest"
    }
]

evaluation_results = []

for item in evaluation_questions:

    result = unified_answer(item["question"])

    answer_text = result["answer"].lower()
    expected_text = item["expected"].lower()

    correct = expected_text in answer_text

    citation_check = validate_citations(result)

    evaluation_results.append({
        "question": item["question"],
        "expected": item["expected"],
        "correct": correct,
        "citation_coverage": citation_check["citation_coverage"],
        "evidence_count": len(result["evidence"])
    })


total_questions = len(evaluation_results)

correct_answers = sum(
    item["correct"]
    for item in evaluation_results
)

accuracy = (
    correct_answers / total_questions
    if total_questions
    else 0
)

average_citation_coverage = (
    sum(
        item["citation_coverage"]
        for item in evaluation_results
    ) / total_questions
    if total_questions
    else 0
)


print("=" * 70)
print("RAG EVALUATION RESULTS")
print("=" * 70)

for item in evaluation_results:

    print("\nQuestion:")
    print(item["question"])

    print("Expected:", item["expected"])
    print("Correct:", item["correct"])

    print(
        "Citation coverage:",
        f"{item['citation_coverage']:.2%}"
    )

    print(
        "Evidence count:",
        item["evidence_count"]
    )

print("\n" + "=" * 70)

print(
    "Answer accuracy:",
    f"{accuracy:.2%}"
)

print(
    "Average citation coverage:",
    f"{average_citation_coverage:.2%}"
)

RAG EVALUATION RESULTS

Question:
Which component consumes BatteryTemperature?
Expected: ThermalController
Correct: True
Citation coverage: 100.00%
Evidence count: 1

Question:
Which components depend on BatteryMonitor?
Expected: ThermalController
Correct: True
Citation coverage: 100.00%
Evidence count: 3

Question:
What interface connects BatteryMonitor and Dashboard?
Expected: BatteryStatusInterface
Correct: True
Citation coverage: 100.00%
Evidence count: 1

Question:
Which component provides BatteryStatus?
Expected: BatteryMonitor
Correct: True
Citation coverage: 100.00%
Evidence count: 3

Question:
What is the functional flow from battery measurement to cooling request?
Expected: CoolingRequest
Correct: True
Citation coverage: 100.00%
Evidence count: 6

Answer accuracy: 100.00%
Average citation coverage: 100.00%


In [416]:
# Cell 46

def check_architecture_consistency(architecture):
    """
    Check basic consistency rules across the extracted
    AUTOSAR architecture knowledge.
    """

    issues = []

    components = architecture.get(
        "components",
        []
    )

    interfaces = architecture.get(
        "interfaces",
        []
    )

    data_flows = architecture.get(
        "data_flows",
        []
    )

    dependencies = architecture.get(
        "dependencies",
        []
    )

    component_names = {
        str(component.get("name"))
        for component in components
    }

    # --------------------------------------------------
    # Rule 1: Interface provider/consumer exists
    # --------------------------------------------------

    for interface in interfaces:

        provider = interface.get(
            "provider"
        )

        consumer = interface.get(
            "consumer"
        )

        name = interface.get(
            "name"
        )

        if provider not in component_names:

            issues.append({
                "type":
                    "missing_interface_provider",

                "interface":
                    name,

                "message":
                    f"Provider '{provider}' "
                    f"for interface '{name}' "
                    f"is not present in component inventory."
            })

        # Consumer may legitimately be an external
        # system such as CoolingSystem.
        # Therefore we report it rather than
        # automatically treating it as an error.

        if consumer and consumer not in component_names:

            issues.append({
                "type":
                    "external_interface_consumer",

                "interface":
                    name,

                "message":
                    f"Consumer '{consumer}' "
                    f"for interface '{name}' "
                    f"is outside the current "
                    f"component inventory."
            })

    # --------------------------------------------------
    # Rule 2: Dependency references
    # --------------------------------------------------

    for dependency in dependencies:

        dependent = dependency.get(
            "dependent_component"
        )

        dependency_component = dependency.get(
            "dependency_component"
        )

        if dependent not in component_names:

            issues.append({
                "type":
                    "missing_dependent_component",

                "message":
                    f"Dependency references "
                    f"unknown component "
                    f"'{dependent}'."
            })

        if dependency_component not in component_names:

            issues.append({
                "type":
                    "missing_dependency_component",

                "message":
                    f"Dependency target "
                    f"'{dependency_component}' "
                    f"is not present in "
                    f"component inventory."
            })

    # --------------------------------------------------
    # Rule 3: Data-flow component references
    # --------------------------------------------------

    for flow in data_flows:

        component = flow.get(
            "component"
        )

        if component not in component_names:

            issues.append({
                "type":
                    "missing_data_flow_component",

                "message":
                    f"Data flow for signal "
                    f"'{flow.get('signal')}' "
                    f"references unknown component "
                    f"'{component}'."
            })

    return issues


consistency_issues = check_architecture_consistency(
    architecture_knowledge
)

print("=" * 70)
print("ARCHITECTURE CONSISTENCY CHECK")
print("=" * 70)

print(
    "Issues found:",
    len(consistency_issues)
)

for issue in consistency_issues:

    print(
        f"\n[{issue['type']}]"
    )

    print(
        issue["message"]
    )

ARCHITECTURE CONSISTENCY CHECK
Issues found: 2

[external_interface_consumer]
Consumer 'CoolingSystem' for interface 'CoolingRequestInterface' is outside the current component inventory.

[missing_dependent_component]
Dependency references unknown component 'CoolingSystem'.


In [418]:
# Cell 47

def compare_architecture_revisions(
    old_architecture,
    new_architecture
):
    """
    Compare two extracted architecture knowledge
    structures and report additions/removals.
    """

    categories = [
        "components",
        "interfaces",
        "data_flows",
        "dependencies",
        "functional_flows"
    ]

    comparison = {}

    for category in categories:

        old_items = old_architecture.get(
            category,
            []
        )

        new_items = new_architecture.get(
            category,
            []
        )

        old_serialized = {
            json.dumps(
                item,
                sort_keys=True
            ): item
            for item in old_items
        }

        new_serialized = {
            json.dumps(
                item,
                sort_keys=True
            ): item
            for item in new_items
        }

        added = [
            new_serialized[key]
            for key in new_serialized
            if key not in old_serialized
        ]

        removed = [
            old_serialized[key]
            for key in old_serialized
            if key not in new_serialized
        ]

        comparison[category] = {
            "added": added,
            "removed": removed,
            "added_count": len(added),
            "removed_count": len(removed)
        }

    return comparison


# Self-comparison sanity test
comparison_result = compare_architecture_revisions(
    architecture_knowledge,
    architecture_knowledge
)

print("=" * 70)
print("REVISION COMPARISON SANITY TEST")
print("=" * 70)

for category, result in comparison_result.items():

    print(
        f"{category}: "
        f"+{result['added_count']} "
        f"/ -{result['removed_count']}"
    )

REVISION COMPARISON SANITY TEST
components: +0 / -0
interfaces: +0 / -0
data_flows: +0 / -0
dependencies: +0 / -0
functional_flows: +0 / -0


In [419]:
# Cell 48

import csv

EXPORT_DIR = OUTPUT_DIR / "structured_export"

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------
# JSON export
# --------------------------------------------------

json_export_path = (
    EXPORT_DIR
    / "architecture_knowledge_export.json"
)

with open(
    json_export_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        architecture_knowledge,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# Component CSV
# --------------------------------------------------

components_csv_path = (
    EXPORT_DIR
    / "components.csv"
)

with open(
    components_csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "name",
            "type",
            "role",
            "inputs",
            "outputs"
        ]
    )

    writer.writeheader()

    for component in architecture_knowledge.get(
        "components",
        []
    ):

        writer.writerow({
            "name":
                component.get("name"),

            "type":
                component.get("type"),

            "role":
                component.get("role"),

            "inputs":
                ", ".join(
                    component.get(
                        "inputs",
                        []
                    )
                ),

            "outputs":
                ", ".join(
                    component.get(
                        "outputs",
                        []
                    )
                )
        })


# --------------------------------------------------
# Interface CSV
# --------------------------------------------------

interfaces_csv_path = (
    EXPORT_DIR
    / "interfaces.csv"
)

with open(
    interfaces_csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "name",
            "provider",
            "consumer",
            "data_elements"
        ]
    )

    writer.writeheader()

    for interface in architecture_knowledge.get(
        "interfaces",
        []
    ):

        writer.writerow({
            "name":
                interface.get("name"),

            "provider":
                interface.get("provider"),

            "consumer":
                interface.get("consumer"),

            "data_elements":
                ", ".join(
                    interface.get(
                        "data_elements",
                        []
                    )
                )
        })


print("=" * 70)
print("STRUCTURED EXPORT COMPLETED")
print("=" * 70)

print(
    "JSON:",
    json_export_path
)

print(
    "Components CSV:",
    components_csv_path
)

print(
    "Interfaces CSV:",
    interfaces_csv_path
)

STRUCTURED EXPORT COMPLETED
JSON: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\structured_export\architecture_knowledge_export.json
Components CSV: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\structured_export\components.csv
Interfaces CSV: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\structured_export\interfaces.csv


In [420]:
# Cell 49

report_path = (
    REPORTS_DIR
    / "architecture_analysis_report.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "AUTOSAR HLD ARCHITECTURE ANALYSIS REPORT\n"
    )

    report.write(
        "=" * 60 + "\n\n"
    )

    report.write(
        f"Document: {document_name}\n\n"
    )

    report.write(
        "ARCHITECTURE INVENTORY\n"
    )

    report.write(
        "-" * 60 + "\n"
    )

    for category in [
        "components",
        "ports",
        "interfaces",
        "data_flows",
        "functional_flows",
        "dependencies",
        "architecture_relationships"
    ]:

        count = len(
            architecture_knowledge.get(
                category,
                []
            )
        )

        report.write(
            f"{category}: {count}\n"
        )

    report.write(
        "\nCONSISTENCY CHECK\n"
    )

    report.write(
        "-" * 60 + "\n"
    )

    report.write(
        f"Issues detected: "
        f"{len(consistency_issues)}\n"
    )

    for issue in consistency_issues:

        report.write(
            f"- {issue['message']}\n"
        )

    report.write(
        "\nRAG EVALUATION\n"
    )

    report.write(
        "-" * 60 + "\n"
    )

    report.write(
        f"Answer accuracy: "
        f"{accuracy:.2%}\n"
    )

    report.write(
        f"Average citation coverage: "
        f"{average_citation_coverage:.2%}\n"
    )


print("=" * 70)
print("ARCHITECTURE REPORT CREATED")
print("=" * 70)

print(
    "Report:",
    report_path
)

ARCHITECTURE REPORT CREATED
Report: c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\reports\architecture_analysis_report.txt


In [422]:
# Cell 50

print("=" * 70)
print("AUTOSAR HLD DOCUMENT ANALYSIS ASSISTANT")
print("FINAL SYSTEM HEALTH CHECK")
print("=" * 70)

checks = {
    "Architecture knowledge loaded":
        bool(architecture_knowledge),

    "RAG documents created":
        len(rag_documents) > 0,

    "FAISS index created":
        faiss_index.ntotal > 0,

    "Semantic search available":
        callable(semantic_search),

    "Structured retrieval available":
        callable(combined_retrieval),

    "Evidence preparation available":
        callable(prepare_evidence),

    "Citation validation available":
        callable(validate_citations),

    "Architecture consistency checker":
        callable(check_architecture_consistency),

    "Revision comparison available":
        callable(compare_architecture_revisions),

    "Structured export created":
        json_export_path.exists(),

    "Architecture report created":
        report_path.exists(),

    "RAG answer accuracy >= 80%":
        accuracy >= 0.80,

    "Citation coverage >= 80%":
        average_citation_coverage >= 0.80
}


passed = 0

for check_name, status in checks.items():

    symbol = "PASS" if status else "FAIL"

    print(
        f"[{symbol}] {check_name}"
    )

    if status:
        passed += 1


print("\n" + "=" * 70)

print(
    f"System checks passed: "
    f"{passed}/{len(checks)}"
)

print(
    "RAG accuracy:",
    f"{accuracy:.2%}"
)

print(
    "Citation coverage:",
    f"{average_citation_coverage:.2%}"
)

print(
    "Architecture consistency issues:",
    len(consistency_issues)
)

print("=" * 70)

if passed == len(checks):

    print(
        "AUTOSAR HLD ASSISTANT "
        "BASELINE IS HEALTHY."
    )

else:

    print(
        "Some checks require review."
    )

AUTOSAR HLD DOCUMENT ANALYSIS ASSISTANT
FINAL SYSTEM HEALTH CHECK
[PASS] Architecture knowledge loaded
[PASS] RAG documents created
[PASS] FAISS index created
[PASS] Semantic search available
[PASS] Structured retrieval available
[PASS] Evidence preparation available
[PASS] Citation validation available
[PASS] Architecture consistency checker
[PASS] Revision comparison available
[PASS] Structured export created
[PASS] Architecture report created
[PASS] RAG answer accuracy >= 80%
[PASS] Citation coverage >= 80%

System checks passed: 13/13
RAG accuracy: 100.00%
Citation coverage: 100.00%
Architecture consistency issues: 2
AUTOSAR HLD ASSISTANT BASELINE IS HEALTHY.
